In [3]:
import os
import torch
import numpy as np
import onnxruntime as ort
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.datasets import ImageFolder
from transformers import ConvNextImageProcessor
from huggingface_hub import hf_hub_download
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from base_fusion_model import FusionClassifier
from base_model_resnet import Car_Classifier_Resnet

# ==========================================
# IMPORT YOUR BASE PYTORCH MODELS HERE
# Example: from my_models import FusionClassifier, Car_Classifier_Resnet
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HF_TOKEN = os.getenv("HF_TOKEN")  # Ensure your Hugging Face token is set in environment

# ==========================================
# 1. IMAGE PROCESSING & TRANSFORMS
# ==========================================

# --- Fusion Model Transforms ---
IMG_SIZE = 260
convnext_model_name = "facebook/convnext-small-224"
processor = ConvNextImageProcessor.from_pretrained(convnext_model_name)

fusion_val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
])

eff_normalize = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# --- ResNet Model Transforms ---
resnet_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


# ==========================================
# 2. DATASETS
# ==========================================

class FusionDataset(Dataset):
    def __init__(self, data_dir, processor, transform=None):
        self.dataset = ImageFolder(root=data_dir, transform=transform)
        self.processor = processor
        self.classes = self.dataset.classes

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        pil_image, label = self.dataset[idx]

        # EfficientNet branch
        pixel_eff = eff_normalize(pil_image)

        # ConvNeXt branch
        enc = self.processor(images=pil_image, return_tensors='pt')
        pixel_cnx = enc['pixel_values'].squeeze(0)

        return {
            'pixel_values_eff': pixel_eff,
            'pixel_values_cnx': pixel_cnx,
            'labels': torch.tensor(label)
        }

class ResNetDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.dataset = ImageFolder(root=data_dir, transform=transform)
        self.classes = self.dataset.classes

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]


# ==========================================
# 3. HUGGING FACE MODEL LOADER
# ==========================================
def load_models_from_hf(repo_id, pt_filename, onnx_filename, model_instance):
    """
    Downloads PyTorch and ONNX models from HF and loads PyTorch weights into the base instance.
    """
    print(f"\n⬇️ Downloading models from Hugging Face: {repo_id}")
    
    # 1. Download and load PyTorch weights
    pt_path = hf_hub_download(repo_id=repo_id, filename=pt_filename, token=HF_TOKEN)
    checkpoint = torch.load(pt_path, map_location=device)
    
    # Handle both full checkpoints dicts and direct state_dicts
    if 'model_state_dict' in checkpoint:
        model_instance.load_state_dict(checkpoint['model_state_dict'])
        print(f"✅ PyTorch model loaded (Val Acc: {checkpoint.get('val_acc', 'N/A')}%)")
    else:
        model_instance.load_state_dict(checkpoint)
        print("✅ PyTorch model loaded")
        
    model_instance.to(device)
    model_instance.eval()

    # 2. Download ONNX model
    onnx_path = hf_hub_download(repo_id=repo_id, filename=onnx_filename, token=HF_TOKEN)
    print("✅ ONNX model downloaded")
    
    return model_instance, onnx_path


# ==========================================
# 4. EVALUATION ENGINE
# ==========================================
def evaluate_models(model_type, torch_model, onnx_model_path, dataloader, target_names):
    """
    Runs evaluation for both PyTorch and ONNX models side-by-side.
    model_type must be either 'fusion' or 'resnet'.
    """
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    ort_session = ort.InferenceSession(onnx_model_path, providers=providers)
    
    all_labels = []
    torch_preds = []
    onnx_preds = []

    print(f"\n🚀 Starting Evaluation for {model_type.upper()} Model...")
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            if model_type == 'fusion':
                images_eff = batch['pixel_values_eff'].to(device)
                images_cnx = batch['pixel_values_cnx'].to(device)
                labels = batch['labels'].numpy()
                
                # PyTorch Inference
                logits_torch = torch_model(images_eff, images_cnx)
                
                # ONNX Inference (Requires inputs matching the exported graph)
                onnx_inputs = {
                    ort_session.get_inputs()[0].name: images_eff.cpu().numpy(),
                    ort_session.get_inputs()[1].name: images_cnx.cpu().numpy()
                }
                logits_onnx = ort_session.run(None, onnx_inputs)[0]

            elif model_type == 'resnet':
                images, labels_tensor = batch
                images = images.to(device)
                labels = labels_tensor.numpy()
                
                # PyTorch Inference
                logits_torch = torch_model(images)
                
                # ONNX Inference
                onnx_inputs = {ort_session.get_inputs()[0].name: images.cpu().numpy()}
                logits_onnx = ort_session.run(None, onnx_inputs)[0]

            # Collect Predictions
            preds_t = torch.argmax(logits_torch, dim=1).cpu().numpy()
            preds_o = np.argmax(logits_onnx, axis=1)

            all_labels.extend(labels)
            torch_preds.extend(preds_t)
            onnx_preds.extend(preds_o)

    # Helper function to print detailed metrics
    def print_metrics(y_true, y_pred, title):
        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
        rec = recall_score(y_true, y_pred, average='macro', zero_division=0)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

        print(f"\n{'='*50}")
        print(f" {title.center(48)} ")
        print(f"{'='*50}")
        print(f"Accuracy  : {acc:.4f}")
        print(f"Precision : {prec:.4f}")
        print(f"Recall    : {rec:.4f}")
        print(f"F1 Score  : {f1:.4f}")
        print("-" * 50)
        print("Classification Report:")
        print(classification_report(y_true, y_pred, target_names=target_names, zero_division=0))

    print_metrics(all_labels, torch_preds, f"{model_type.upper()} - PYTORCH MODEL")
    print_metrics(all_labels, onnx_preds, f"{model_type.upper()} - ONNX MODEL")

In [5]:
if __name__ == "__main__":
    eval_dir = "D:\\DamageLensAI\\val"  # Adjust to your evaluation folder

    # ---------------------------------------------------------
    # EVALUATE FUSION MODEL
    # ---------------------------------------------------------
    fusion_dataset = FusionDataset(data_dir=eval_dir, processor=processor, transform=fusion_val_transforms)
    fusion_loader = DataLoader(fusion_dataset, batch_size=16, shuffle=False)
    
    # Initialize your base Fusion PyTorch architecture
    # Replace 'FusionClassifier' with your actual class name and arguments
    base_fusion = FusionClassifier(num_classes=len(fusion_dataset.classes), convnext_model_name=convnext_model_name)
    
    # Load weights and get ONNX path from Hub
    torch_fusion, onnx_fusion_path = load_models_from_hf(
        repo_id="junaid17/best_fusion_model_fp16",
        pt_filename="best_fusion_model_fp16.pt", # Update if named differently on HF
        onnx_filename="fusion_model.onnx",
        model_instance=base_fusion
    )
    
    evaluate_models('fusion', torch_fusion, onnx_fusion_path, fusion_loader, fusion_dataset.classes)

    # ---------------------------------------------------------
    # EVALUATE RESNET MODEL
    # ---------------------------------------------------------
    resnet_dataset = ResNetDataset(data_dir=eval_dir, transform=resnet_test_transforms)
    resnet_loader = DataLoader(resnet_dataset, batch_size=16, shuffle=False)
    
    # Initialize your base ResNet PyTorch architecture
    # Replace 'get_resnet_model' with how you initialize your ResNet
    base_resnet = Car_Classifier_Resnet(num_classes=len(resnet_dataset.classes))
    
    # Load weights and get ONNX path from Hub
    torch_resnet, onnx_resnet_path = load_models_from_hf(
        repo_id="junaid17/car-damage-classifier",
        pt_filename="car-damage-classifier.pt", # Update if named differently on HF
        onnx_filename="car-damage-classifier.onnx",
        model_instance=base_resnet
    )
    
    evaluate_models('resnet', torch_resnet, onnx_resnet_path, resnet_loader, resnet_dataset.classes)

Loading weights:   0%|          | 0/342 [00:00<?, ?it/s]

[transformers] ConvNextModel LOAD REPORT from: facebook/convnext-small-224
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



⬇️ Downloading models from Hugging Face: junaid17/best_fusion_model_fp16
✅ PyTorch model loaded
✅ ONNX model downloaded


d:\DamageLensAI\myvenv\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(



🚀 Starting Evaluation for FUSION Model...


...

✅ PyTorch model loaded
✅ ONNX model downloaded


d:\DamageLensAI\myvenv\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(



🚀 Starting Evaluation for RESNET Model...


Evaluating: 100%|██████████| 29/29 [00:35<00:00,  1.21s/it]


              RESNET - PYTORCH MODEL              
Accuracy  : 0.7739
Precision : 0.7734
Recall    : 0.7661
F1 Score  : 0.7681
--------------------------------------------------
Classification Report:
              precision    recall  f1-score   support

  F_Breakage       0.83      0.78      0.80       100
   F_Crushed       0.63      0.75      0.69        80
    F_Normal       0.89      0.85      0.87       100
  R_Breakage       0.81      0.72      0.76        60
   R_Crushed       0.67      0.72      0.69        60
    R_Normal       0.81      0.78      0.80        60

    accuracy                           0.77       460
   macro avg       0.77      0.77      0.77       460
weighted avg       0.78      0.77      0.78       460


               RESNET - ONNX MODEL                
Accuracy  : 0.7739
Precision : 0.7734
Recall    : 0.7661
F1 Score  : 0.7681
--------------------------------------------------
Classification Report:
              precision    recall  f1-score   support